# TruthLens — Train the fake-news classifier

Run this on **Google Colab** (Runtime → Run all).

Dataset: [Fake and Real News Dataset (Kaggle)](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset) — download `Fake.csv` and `True.csv`.

Output: `fake_news_model.joblib` → put it in the project's `models/` folder.
Label convention: **1 = real, 0 = fake**.

In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

## 1. Upload the data

In [ ]:
# In Colab this opens a file picker — select Fake.csv and True.csv
try:
    from google.colab import files
    files.upload()
except ImportError:
    print("Not on Colab — make sure Fake.csv and True.csv are in this folder.")

In [ ]:
fake = pd.read_csv("Fake.csv")
real = pd.read_csv("True.csv")
fake["label"] = 0
real["label"] = 1

# Real articles in this dataset start with "CITY (Reuters) -", which would let the
# model cheat. Strip it so it learns from the content instead.
real["text"] = real["text"].str.replace(r"^.*?\(Reuters\)\s*-\s*", "", regex=True)

df = pd.concat([fake, real], ignore_index=True)
df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")
df = df[df["content"].str.split().str.len() > 20].drop_duplicates("content")

print(df["label"].value_counts())
df.head()

## 2. Train

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["content"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_df=0.7, min_df=3,
                              ngram_range=(1, 2), max_features=100_000, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, C=2.0)),
])
model.fit(X_train, y_train)

## 3. Evaluate

In [ ]:
pred = model.predict(X_test)
print(classification_report(y_test, pred, target_names=["fake", "real"]))
print(confusion_matrix(y_test, pred))

In [ ]:
# Quick sanity check on your own sentences
samples = [
    "The central bank raised interest rates by a quarter point on Wednesday, citing persistent inflation.",
    "SHOCKING: Doctors don't want you to know this one weird trick that cures everything!!!",
]
for s, p in zip(samples, model.predict_proba(samples)[:, 1]):
    print(f"{p:.2f} real  |  {s}")

## 4. Save and download

In [ ]:
joblib.dump(model, "fake_news_model.joblib", compress=3)

try:
    from google.colab import files
    files.download("fake_news_model.joblib")
except ImportError:
    print("Saved fake_news_model.joblib — move it into the models/ folder.")